In [18]:
import dagshub
import mlflow
import tensorflow as tf
from pathlib import Path
import os
from cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from cnnClassifier.utils.common import read_yaml, create_directories, save_json
from dataclasses import dataclass
import mlflow.tensorflow
from urllib.parse import urlparse

In [ ]:
dagshub.init(repo_owner='gundesanskar71', repo_name='Kidney-disease-Classification-', mlflow=True)

In [6]:
# Load the model
model = tf.keras.models.load_model("artifacts/training/model.h5")

I0000 00:00:1780655566.774571   84598 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4164 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [15]:
@dataclass(frozen=True)
class EvalConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    mlflow_uri: str
    params_image_size: list
    params_batch_size: int

In [17]:
class ConfigurationManager:
    def __init__(self, config_filepath=str(CONFIG_FILE_PATH), params_filepath=str(PARAMS_FILE_PATH)):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_eval_config(self) -> EvalConfig:
        eval_config = EvalConfig(
            path_of_model="artifacts/training/model.h5",
            training_data="artifacts/data_ingestion/kidney_ct_scan_dataset/kidney_ct_scan_dataset",
            mlflow_uri="https://dagshub.com/gundesanskar71/Kidney-disease-Classification-.mlflow",
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE
        )

        return eval_config

In [27]:
class Evaluation:
    def __init__(self, config: EvalConfig):
        self.config = config

    def _valid_generator(self):
    
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )
        
        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )
        
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)    
    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)

    def evaluate(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)
        self.save_score()

    def log_into_mlflow(self):
        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        # start the mlflow and log the artifacts
        with mlflow.start_run():
            mlflow.log_params(self.config.all_params)
            mlflow.log_metrics({"loss": self.score[0], "accuracy": self.score[1]})


            # model registry does not work file store
            if tracking_url_type_store != "file":
                mlflow.tensorflow.log_model(self.model, "model", registered_model_name="VGG16Model")
            else:
                mlflow.tensorflow.log_model(self.model, "model")

In [31]:
try:
    config = ConfigurationManager()
    eval_config = config.get_eval_config()
    obj = Evaluation(eval_config)
    obj.evaluate()
    obj.log_into_mlflow()
    print("done")
except Exception as e:
    raise e

Found 3732 images belonging to 4 classes.
234/234 ━━━━━━━━━━━━━━━━━━━━ 32s 134ms/step - accuracy: 0.5340 - loss: 8.3748


2026/06/05 17:45:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/05 17:45:33 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
Successfully registered model 'VGG16Model'.
2026/06/05 17:47:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: VGG16Model, version 1
Created version '1' of model 'VGG16Model'.


🏃 View run polite-roo-140 at: https://dagshub.com/gundesanskar71/Kidney-disease-Classification-.mlflow/#/experiments/0/runs/a3bf7d607d534059ac85ec808accaa64
🧪 View experiment at: https://dagshub.com/gundesanskar71/Kidney-disease-Classification-.mlflow/#/experiments/0
done
